# Введение в MapReduce модель на Python


In [1]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [2]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [3]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [4]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [5]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [6]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [7]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [8]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [9]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [10]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [11]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [12]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [13]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [14]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [15]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(1.0855333507737495)),
 (1, np.float64(1.0855333507737495)),
 (2, np.float64(1.0855333507737495)),
 (3, np.float64(1.0855333507737495)),
 (4, np.float64(1.0855333507737495))]

## Inverted index

In [16]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [17]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [18]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [19]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('banana', 2), ('is', 18)]),
 (1, [('a', 2), ('it', 18), ('what', 10)])]

## TeraSort

In [20]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.021916989463378056)),
   (None, np.float64(0.04849259334564615)),
   (None, np.float64(0.06247662980499025)),
   (None, np.float64(0.06618663119861945)),
   (None, np.float64(0.07613893349187772)),
   (None, np.float64(0.07939072322231955)),
   (None, np.float64(0.12924173419474616)),
   (None, np.float64(0.1301726609868511)),
   (None, np.float64(0.15633779089052469)),
   (None, np.float64(0.19052964645065984)),
   (None, np.float64(0.20668384390112304)),
   (None, np.float64(0.23286413812078932)),
   (None, np.float64(0.24364320080029644)),
   (None, np.float64(0.2989573608979871)),
   (None, np.float64(0.3289646178617768)),
   (None, np.float64(0.3552725902011865)),
   (None, np.float64(0.4584947506772957))]),
 (1,
  [(None, np.float64(0.5314871325152853)),
   (None, np.float64(0.5419696365800601)),
   (None, np.float64(0.5559278620459162)),
   (None, np.float64(0.5669540523131674)),
   (None, np.float64(0.588361707213039)),
   (None, np.float64(0.6494425

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

In [23]:
from typing import Iterator

input_numbers_avg = [10, 5, 20, 15, 25, 8]

def RECORDREADER_AVG():
  for num in input_numbers_avg:
    yield (None, num)

def MAP_AVG(key, value):
  yield ('sum_and_count', (value, 1))

def REDUCE_AVG(key: str, values: Iterator[tuple]):
  total_sum = 0
  total_count = 0
  for val, count in values:
    total_sum += val
    total_count += count
  if total_count > 0:
    yield (key, total_sum / total_count)
  else:
    yield (key, 0)

output_avg = MapReduce(RECORDREADER_AVG, MAP_AVG, REDUCE_AVG)
output_avg = list(output_avg)
print(output_avg)

[('sum_and_count', 13.833333333333334)]


### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [21]:
from typing import Iterator

input_numbers = [10, 5, 20, 15, 25, 8]

def RECORDREADER():
  for num in input_numbers:
    yield (None, num)

def MAP(key, value):
  yield ('max_value', value)

def REDUCE(key: str, values: Iterator[int]):
  max_val = -float('inf')
  for v in values:
    if v > max_val:
      max_val = v
  yield (key, max_val)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
print(output)

[('max_value', 25)]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [22]:
from typing import Iterator

input_numbers_avg = [10, 5, 20, 15, 25, 8]

def RECORDREADER_AVG():
  for num in input_numbers_avg:
    yield (None, num)

def MAP_AVG(key, value):
  yield ('sum_and_count', (value, 1))

def REDUCE_AVG(key: str, values: Iterator[tuple]):
  total_sum = 0
  total_count = 0
  for val, count in values:
    total_sum += val
    total_count += count
  if total_count > 0:
    yield (key, total_sum / total_count)
  else:
    yield (key, 0)

output_avg = MapReduce(RECORDREADER_AVG, MAP_AVG, REDUCE_AVG)
output_avg = list(output_avg)
print(output_avg)

[('sum_and_count', 13.833333333333334)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [44]:
from typing import Iterator, List, Tuple, Any

def groupbykey_sorted(iterable: List[Tuple[Any, Any]]) -> List[Tuple[Any, List[Any]]]:
  sorted_iterable = sorted(iterable, key=lambda x: x[0])

  if not sorted_iterable:
    return []

  results = []
  current_key = None
  current_values = []

  for key, value in sorted_iterable:
    if current_key is None or key != current_key:
      if current_key is not None:
        results.append((current_key, current_values))
      current_key = key
      current_values = [value]
    else:
      current_values.append(value)

  if current_key is not None:
    results.append((current_key, current_values))

  return results

shuffle_output_sorted = groupbykey_sorted(map_output)
print("Результат groupbykey_sorted:", shuffle_output_sorted)

# Для сравнения, исходный groupbykey:
shuffle_output_original = groupbykey(map_output)
print("Результат оригинального groupbykey:", list(shuffle_output_original))

Результат groupbykey_sorted: [(25, [User(id=1, age=25, social_contacts=240, gender='female'), User(id=2, age=25, social_contacts=500, gender='female')]), (33, [User(id=3, age=33, social_contacts=800, gender='female')])]
Результат оригинального groupbykey: [(25, [User(id=1, age=25, social_contacts=240, gender='female'), User(id=2, age=25, social_contacts=500, gender='female')]), (33, [User(id=3, age=33, social_contacts=800, gender='female')])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [45]:
from typing import Iterator, Any

input_data_with_duplicates = [
    ('a', 1), ('b', 2), ('a', 1), ('c', 3), ('b', 2), ('d', 4), ('a', 5)
]

def RECORDREADER_DEDUP():
  for item in input_data_with_duplicates:
    yield (item, None)

def MAP_DEDUP(key: Any, value: Any):
  yield (key, None)

def REDUCE_DEDUP(key: Any, values: Iterator[Any]):
  yield (key, None)

output_dedup = MapReduce(RECORDREADER_DEDUP, MAP_DEDUP, REDUCE_DEDUP)
output_dedup = list(output_dedup)

unique_elements = [key for key, _ in output_dedup]
print("Исходные данные с дубликатами:", input_data_with_duplicates)
print("Уникальные элементы (без дубликатов):", sorted(unique_elements))

input_numbers_dedup = [1, 5, 2, 1, 5, 3, 2, 4, 1]
def RECORDREADER_NUM_DEDUP():
  for num in input_numbers_dedup:
    yield (num, None)

output_num_dedup = MapReduce(RECORDREADER_NUM_DEDUP, MAP_DEDUP, REDUCE_DEDUP)
unique_numbers = sorted([key for key, _ in list(output_num_dedup)])
print("\nИсходные числа с дубликатами:", input_numbers_dedup)
print("Уникальные числа (без дубликатов):", unique_numbers)

Исходные данные с дубликатами: [('a', 1), ('b', 2), ('a', 1), ('c', 3), ('b', 2), ('d', 4), ('a', 5)]
Уникальные элементы (без дубликатов): [('a', 1), ('a', 5), ('b', 2), ('c', 3), ('d', 4)]

Исходные числа с дубликатами: [1, 5, 2, 1, 5, 3, 2, 4, 1]
Уникальные числа (без дубликатов): [1, 2, 3, 4, 5]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [46]:
from typing import NamedTuple, Iterator

def predicate_C(row: User) -> bool:
  return row.gender == 'female'

def RECORDREADER_SELECTION():
  return [(u.id, u) for u in input_collection]

def MAP_SELECTION(key: int, row: User):
  if predicate_C(row):
    yield (row, row) # Ключ и значение одинаковы, равны t

def REDUCE_SELECTION(key: User, values: Iterator[User]):
  yield (key, key)

# Выполняем MapReduce
output_selection = MapReduce(RECORDREADER_SELECTION, MAP_SELECTION, REDUCE_SELECTION)
output_selection = list(output_selection)
print("Результат выборки (женщины):")
for item in output_selection:
  print(item[0]) # Выводим только кортеж t


Результат выборки (женщины):
User(id=1, age=25, social_contacts=240, gender='female')
User(id=2, age=25, social_contacts=500, gender='female')
User(id=3, age=33, social_contacts=800, gender='female')


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [47]:
from typing import NamedTuple, Iterator, Any

projection_attributes = ['id', 'age', 'gender']

def RECORDREADER_PROJECTION():
  return [(u.id, u) for u in input_collection]

def MAP_PROJECTION(key: int, row: User):
  projected_data = {attr: getattr(row, attr) for attr in projection_attributes}

  t_prime = tuple(projected_data.values())
  yield (t_prime, t_prime)

def REDUCE_PROJECTION(key: Any, values: Iterator[Any]):
    yield (key, key)

# Выполняем MapReduce
output_projection = MapReduce(RECORDREADER_PROJECTION, MAP_PROJECTION, REDUCE_PROJECTION)
output_projection = list(output_projection)
print(f"Результат проекции на атрибуты {projection_attributes}:")
for item in output_projection:
  print(item[0]) # Выводим только спроецированный кортеж t'

Результат проекции на атрибуты ['id', 'age', 'gender']:
(0, 55, 'male')
(1, 25, 'female')
(2, 25, 'female')
(3, 33, 'female')


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [48]:
from typing import Iterator, Any

R_union = [('a', 1), ('b', 2), ('c', 3)]
S_union = [('b', 2), ('d', 4), ('e', 5)]

def RECORDREADER_UNION():
  for t in R_union:
    yield (t, t)
  for t in S_union:
    yield (t, t)

def MAP_UNION(key: Any, value: Any):
  yield (key, value)

def REDUCE_UNION(key: Any, values: Iterator[Any]):
  yield (key, key)

output_union = MapReduce(RECORDREADER_UNION, MAP_UNION, REDUCE_UNION)
output_union = list(output_union)

print("Отношение R:", R_union)
print("Отношение S:", S_union)
print("Результат Union:", sorted([k for k,v in output_union]))

Отношение R: [('a', 1), ('b', 2), ('c', 3)]
Отношение S: [('b', 2), ('d', 4), ('e', 5)]
Результат Union: [('a', 1), ('b', 2), ('c', 3), ('d', 4), ('e', 5)]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [49]:
from typing import Iterator, Any

R_intersection = [('a', 1), ('b', 2), ('c', 3)]
S_intersection = [('b', 2), ('d', 4), ('e', 5)]

def RECORDREADER_INTERSECTION():
  for t in R_intersection:
    yield (t, t)
  for t in S_intersection:
    yield (t, t)

def MAP_INTERSECTION(key: Any, value: Any):
  yield (key, value)

def REDUCE_INTERSECTION(key: Any, values: Iterator[Any]):
  if len(list(values)) == 2:
    yield (key, key)

output_intersection = MapReduce(RECORDREADER_INTERSECTION, MAP_INTERSECTION, REDUCE_INTERSECTION)
output_intersection = list(output_intersection)

print("Отношение R:", R_intersection)
print("Отношение S:", S_intersection)
print("Результат Intersection:", sorted([k for k,v in output_intersection]))

Отношение R: [('a', 1), ('b', 2), ('c', 3)]
Отношение S: [('b', 2), ('d', 4), ('e', 5)]
Результат Intersection: [('b', 2)]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [50]:
from typing import Iterator, Any

R_diff = [('a', 1), ('b', 2), ('c', 3)]
S_diff = [('b', 2), ('d', 4), ('e', 5)]

def RECORDREADER_DIFFERENCE():
  for t in R_diff:
    yield (t, 'R_tag')
  for t in S_diff:
    yield (t, 'S_tag')

def MAP_DIFFERENCE(key: Any, value: Any):
  yield (key, value)

def REDUCE_DIFFERENCE(key: Any, values: Iterator[Any]):
  values_list = list(values)
  if values_list == ['R_tag']:
    yield (key, key)

output_difference = MapReduce(RECORDREADER_DIFFERENCE, MAP_DIFFERENCE, REDUCE_DIFFERENCE)
output_difference = list(output_difference)

print("Отношение R:", R_diff)
print("Отношение S:", S_diff)
print("Результат Difference (R - S):", sorted([k for k,v in output_difference]))

Отношение R: [('a', 1), ('b', 2), ('c', 3)]
Отношение S: [('b', 2), ('d', 4), ('e', 5)]
Результат Difference (R - S): [('a', 1), ('c', 3)]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [51]:
from typing import Iterator, Any

R_join = [
    ('A1', 10),
    ('A2', 20),
    ('A3', 10)
]

S_join = [
    (10, 'C1'),
    (20, 'C2'),
    (10, 'C3')
]

def RECORDREADER_NATURAL_JOIN():
  for a_val, b_val in R_join:
    yield (b_val, ('R', a_val))
  for b_val, c_val in S_join:
    yield (b_val, ('S', c_val))

def MAP_NATURAL_JOIN(key: Any, value: Any):
  yield (key, value)

def REDUCE_NATURAL_JOIN(b_key: Any, values: Iterator[Any]):
  r_values = [] # Будут хранить a из (R, a)
  s_values = [] # Будут хранить c из (S, c)

  for tag, val in values:
    if tag == 'R':
      r_values.append(val)
    elif tag == 'S':
      s_values.append(val)

  for a_val in r_values:
    for c_val in s_values:
      yield ( (a_val, b_key, c_val), None) # Ключ не нужен, поэтому используем None

output_natural_join = MapReduce(RECORDREADER_NATURAL_JOIN, MAP_NATURAL_JOIN, REDUCE_NATURAL_JOIN)
output_natural_join = list(output_natural_join)

print("Отношение R:", R_join)
print("Отношение S:", S_join)
print("Результат Natural Join:", sorted([k for k,v in output_natural_join]))

Отношение R: [('A1', 10), ('A2', 20), ('A3', 10)]
Отношение S: [(10, 'C1'), (20, 'C2'), (10, 'C3')]
Результат Natural Join: [('A1', 10, 'C1'), ('A1', 10, 'C3'), ('A2', 20, 'C2'), ('A3', 10, 'C1'), ('A3', 10, 'C3')]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [52]:
from typing import Iterator, Any

input_data_grouping = [
    ('group1', 10, 'x'),
    ('group2', 20, 'y'),
    ('group1', 15, 'z'),
    ('group3', 5, 'w'),
    ('group2', 25, 'v'),
    ('group1', 30, 'u')
]

def RECORDREADER_GROUPING():
  for item in input_data_grouping:
    yield (None, item)

def MAP_GROUPING(key: Any, value: Any):
  a, b, _ = value
  yield (a, b)

def REDUCE_GROUPING_SUM(group_key: Any, values: Iterator[Any]):
  total_sum = sum(values)
  yield (group_key, total_sum)

output_grouping_sum = MapReduce(RECORDREADER_GROUPING, MAP_GROUPING, REDUCE_GROUPING_SUM)
output_grouping_sum = list(output_grouping_sum)

print("Исходные данные:", input_data_grouping)
print("Результат группировки и суммирования:", sorted(output_grouping_sum))


def REDUCE_GROUPING_MAX(group_key: Any, values: Iterator[Any]):
  max_value = -float('inf')
  for val in values:
    if val > max_value:
      max_value = val
  yield (group_key, max_value)

output_grouping_max = MapReduce(RECORDREADER_GROUPING, MAP_GROUPING, REDUCE_GROUPING_MAX)
output_grouping_max = list(output_grouping_max)

print("\nРезультат группировки и нахождения максимального значения:", sorted(output_grouping_max))

Исходные данные: [('group1', 10, 'x'), ('group2', 20, 'y'), ('group1', 15, 'z'), ('group3', 5, 'w'), ('group2', 25, 'v'), ('group1', 30, 'u')]
Результат группировки и суммирования: [('group1', 55), ('group2', 45), ('group3', 5)]

Результат группировки и нахождения максимального значения: [('group1', 30), ('group2', 25), ('group3', 5)]


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [53]:
from typing import Iterator, Tuple
import numpy as np

M, N = 5, 4
chunk_width = 2

matrix_data = np.random.rand(M, N)
vector_data = np.random.rand(N)

def data_stream() -> Iterator[Tuple[int, Tuple[int, int, float]]]:
    for chunk_idx in range((N + chunk_width - 1) // chunk_width):
        col_start = chunk_idx * chunk_width
        col_end = min(col_start + chunk_width, N)
        for row_idx in range(M):
            for col_idx in range(col_start, col_end):
                yield chunk_idx, (row_idx, col_idx, matrix_data[row_idx, col_idx])

def map_phase(chunk_idx: int, entry: Tuple[int, int, float]) -> Iterator[Tuple[int, float]]:
    row_idx, col_idx, val = entry
    chunk_vec = vector_data[chunk_idx*chunk_width : chunk_idx*chunk_width + chunk_width]
    result = val * chunk_vec[col_idx - chunk_idx*chunk_width]
    yield row_idx, result

def reduce_phase(row_idx: int, contributions: Iterator[float]) -> Iterator[Tuple[int, float]]:
    yield row_idx, sum(contributions)

def flatten(data):
    for item in data:
        for elem in item:
            yield elem

def groupbykey(items):
    groups = {}
    for k, v in items:
        groups.setdefault(k, []).append(v)
    return groups.items()

stream = list(data_stream())
mapped = list(flatten(map(lambda x: map_phase(*x), stream)))
shuffled = list(groupbykey(mapped))
final = list(flatten(map(lambda x: reduce_phase(*x), shuffled)))

computed = [v for _, v in final]
expected = np.matmul(matrix_data, vector_data)
print(np.allclose(computed, expected))

True


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [ ]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [54]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j, k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(small_mat.shape[0]):
    yield ((i, k), w * small_mat[i][j])

def REDUCE(key, values):
  (i, k) = key
  el_value = 0
  for v in values:
    el_value += v
  yield ((i, k), el_value)

Проверьте своё решение

In [55]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [56]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [57]:
import numpy as np

I, J, K = 2, 3, 40
mat_a = np.random.rand(I, J)
mat_b = np.random.rand(J, K)
expected = np.matmul(mat_a, mat_b)

def data_stream():
    for i in range(mat_a.shape[0]):
        for j in range(mat_a.shape[1]):
            yield ((0, i, j), mat_a[i, j])
    for j in range(mat_b.shape[0]):
        for k in range(mat_b.shape[1]):
            yield ((1, j, k), mat_b[j, k])

def map_join(key, val):
    src, idx1, idx2 = key
    if src == 0:
        yield (idx2, (src, idx1, val))
    else:
        yield (idx1, (src, idx2, val))

def reduce_join(key, vals):
    left = [v for v in vals if v[0] == 0]
    right = [v for v in vals if v[0] == 1]
    for l in left:
        for r in right:
            yield ((l[1], r[1]), l[2] * r[2])

def map_pass(key, val):
    yield (key, val)

def reduce_sum(key, vals):
    yield (key, sum(vals))

def stream_joined():
    for item in joined_result:
        yield item

joined_result = MapReduce(data_stream, map_join, reduce_join)
final_result = MapReduce(stream_joined, map_pass, reduce_sum)

def to_matrix(output):
    out_list = list(output)
    rows = max(i for ((i, k), _) in out_list) + 1
    cols = max(k for ((i, k), _) in out_list) + 1
    result_mat = np.empty((rows, cols))
    for (i_k, val) in out_list:
        result_mat[i_k] = val
    return result_mat

print(np.allclose(expected, to_matrix(final_result)))

True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [58]:
import numpy as np

rows_a, common_dim, cols_b = 2, 3, 40
matrix_a = np.random.rand(rows_a, common_dim)
matrix_b = np.random.rand(common_dim, cols_b)
expected = np.matmul(matrix_a, matrix_b)

def collapse(nested):
    for sub in nested:
        for item in sub:
            yield item

def cluster_by_key(data):
    groups = {}
    for k, v in data:
        groups[k] = groups.get(k, []) + [v]
    return groups.items()

def cluster_distributed(parts, part_func):
    global reducers_count
    buckets = [dict() for _ in range(reducers_count)]
    for part in parts:
        for k, v in part:
            bucket = buckets[part_func(k)]
            bucket[k] = bucket.get(k, []) + [v]
    return [(pid, sorted(b.items(), key=lambda x: x[0])) for pid, b in enumerate(buckets)]

def partition_func(obj):
    global reducers_count
    return hash(obj) % reducers_count

def run_distributed(source_fn, map_fn, reduce_fn, part_fn=partition_func, combine_fn=None):
    mapped = map(lambda reader: collapse(map(lambda kv: map_fn(*kv), reader)), source_fn())
    if combine_fn is not None:
        mapped = map(lambda part: collapse(map(lambda kv: combine_fn(*kv), cluster_by_key(part))), mapped)
    shuffled = cluster_distributed(mapped, part_fn)
    reduced = map(lambda rp: (rp[0], collapse(map(lambda group: reduce_fn(*group), rp[1]))), shuffled)
    print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (_,vs) in collapse([p for (_,p) in shuffled])])))
    return reduced

def to_matrix(output):
    out_list = list(output)
    if not out_list:
        return np.empty((rows_a, cols_b))
    max_i = max(i for ((i,k), _) in out_list) + 1
    max_k = max(k for ((i,k), _) in out_list) + 1
    result = np.empty((max_i, max_k))
    for (ik, val) in out_list:
        result[ik] = val
    return result

def input_source():
    batch_a = []
    for i in range(matrix_a.shape[0]):
        for j in range(matrix_a.shape[1]):
            batch_a.append(((0, i, j), matrix_a[i,j]))
    yield batch_a
    batch_b = []
    for j in range(matrix_b.shape[0]):
        for k in range(matrix_b.shape[1]):
            batch_b.append(((1, j, k), matrix_b[j,k]))
    yield batch_b

def map_join(key, val):
    src, idx1, idx2 = key
    if src == 0:
        yield (idx2, (src, idx1, val))
    else:
        yield (idx1, (src, idx2, val))

def reduce_join(key, vals):
    left = [v for v in vals if v[0] == 0]
    right = [v for v in vals if v[0] == 1]
    for l in left:
        for r in right:
            yield ((l[1], r[1]), l[2] * r[2])

def map_pass(key, val):
    yield (key, val)

def reduce_sum(key, vals):
    total = sum(vals)
    yield (key, total)

maps_count = 4
reducers_count = 2

joined_result = run_distributed(input_source, map_join, reduce_join, part_fn=partition_func, combine_fn=None)

joined_data = []
for pid, partition in joined_result:
    for item in partition:
        joined_data.append(item)

def source_joined():
    yield joined_data

mul_result = run_distributed(source_joined, map_pass, reduce_sum, part_fn=partition_func, combine_fn=None)

solution = []
for pid, partition in mul_result:
    for item in partition:
        solution.append(item)

print(np.allclose(expected, to_matrix(solution)))

126 key-value pairs were sent over a network.
240 key-value pairs were sent over a network.
True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

Да, будет работать при случайном подмножестве элементов, если все ненулевые элементы покрыты, нет дубликатов координат, и сохранены метки 'M'/'N' для различения матриц.

In [59]:
import numpy as np

rows_a, common_dim, cols_b = 2, 3, 40
matrix_a = np.random.rand(rows_a, common_dim)
matrix_b = np.random.rand(common_dim, cols_b)
expected = np.matmul(matrix_a, matrix_b)

def collapse(nested):
    for sub in nested:
        for item in sub:
            yield item

def cluster_by_key(data):
    groups = {}
    for k, v in data:
        groups[k] = groups.get(k, []) + [v]
    return groups.items()

def cluster_distributed(parts, part_func):
    global reducers_count
    buckets = [dict() for _ in range(reducers_count)]
    for part in parts:
        for k, v in part:
            bucket = buckets[part_func(k)]
            bucket[k] = bucket.get(k, []) + [v]
    return [(pid, sorted(b.items(), key=lambda x: x[0])) for pid, b in enumerate(buckets)]

def partition_func(obj):
    global reducers_count
    return hash(obj) % reducers_count

def run_distributed(source_fn, map_fn, reduce_fn, part_fn=partition_func, combine_fn=None):
    mapped = map(lambda reader: collapse(map(lambda kv: map_fn(*kv), reader)), source_fn())
    if combine_fn is not None:
        mapped = map(lambda part: collapse(map(lambda kv: combine_fn(*kv), cluster_by_key(part))), mapped)
    shuffled = cluster_distributed(mapped, part_fn)
    reduced = map(lambda rp: (rp[0], collapse(map(lambda group: reduce_fn(*group), rp[1]))), shuffled)
    print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (_,vs) in collapse([p for (_,p) in shuffled])])))
    return reduced

def to_matrix(output):
    out_list = list(output)
    if not out_list:
        return np.empty((rows_a, cols_b))
    max_i = max(i for ((i,k), _) in out_list) + 1
    max_k = max(k for ((i,k), _) in out_list) + 1
    result = np.empty((max_i, max_k))
    for (ik, val) in out_list:
        result[ik] = val
    return result

def input_source():
    batch_a = []
    for i in range(matrix_a.shape[0]):
        for j in range(matrix_a.shape[1]):
            batch_a.append(((0, i, j), matrix_a[i,j]))
    yield batch_a
    batch_b = []
    for j in range(matrix_b.shape[0]):
        for k in range(matrix_b.shape[1]):
            batch_b.append(((1, j, k), matrix_b[j,k]))
    yield batch_b

def map_join(key, val):
    src, idx1, idx2 = key
    if src == 0:
        yield (idx2, (src, idx1, val))
    else:
        yield (idx1, (src, idx2, val))

def reduce_join(key, vals):
    left = [v for v in vals if v[0] == 0]
    right = [v for v in vals if v[0] == 1]
    for l in left:
        for r in right:
            yield ((l[1], r[1]), l[2] * r[2])

def map_pass(key, val):
    yield (key, val)

def reduce_sum(key, vals):
    total = sum(vals)
    yield (key, total)

maps_count = 4
reducers_count = 2

joined_result = run_distributed(input_source, map_join, reduce_join, part_fn=partition_func, combine_fn=None)

joined_data = []
for pid, partition in joined_result:
    for item in partition:
        joined_data.append(item)

def source_joined():
    yield joined_data

mul_result = run_distributed(source_joined, map_pass, reduce_sum, part_fn=partition_func, combine_fn=None)

solution = []
for pid, partition in mul_result:
    for item in partition:
        solution.append(item)

print(np.allclose(expected, to_matrix(solution)))

126 key-value pairs were sent over a network.
240 key-value pairs were sent over a network.
True
